In [2]:
import os
import mne
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from mne.decoding import CSP

In [3]:
# create folder to store results if not exist
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ2"
if not os.path.exists(result_path):
    os.makedirs(result_path)

# create folder to store plots if not exist
result_plot= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ2/plot"
if not os.path.exists(result_plot):
    os.makedirs(result_plot)


### Load Data

In [5]:
df_skill = pd.read_csv(f"C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/filtered_data_skill.csv")
df_filtered = df_skill[["Participant", "Algorithm", "SkillScore", "EEG", "CrossEEG"]]
df_skill = df_skill[["Participant", "SkillScore"]]
df_skill = df_skill.drop_duplicates()

### assign skill level

quantile = df_skill["SkillScore"].quantile([0.33, 0.66])
lower = quantile[0.33]
upper = quantile[0.66]

df_skill['SkillLevel'] = np.select([df_skill['SkillScore'] < lower, (df_skill['SkillScore'] >= lower) & (df_skill['SkillScore'] <= upper), df_skill['SkillScore'] > upper],
                                 ['Novice', 'Intermediate', 'Expert'],
                                 default='Intermediate')
df_skilled = df_skill.copy()
df_filtered = pd.merge(df_filtered, df_skilled[['Participant', 'SkillLevel']], on='Participant', how='left')
df_filtered = df_filtered[["Participant", "Algorithm", "SkillLevel", "EEG", "CrossEEG"]]
df_filtered

,Participant,Algorithm,SkillLevel,EEG,CrossEEG
0,1,IsPrime,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1,1,SiebDesEratosthenes,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
2,1,IsAnagram,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
3,1,RemoveDoubleChar,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
4,1,BinToDecimal,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
...,...,...,...,...,...
1067,71,DumpSorting,Expert,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1068,71,BinomialCoefficient,Expert,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1069,71,IsAnagram,Expert,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1070,71,ArrayAverage,Expert,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...


### Set Montage

In [10]:
montage_path = result_path + '/../../Master-Thesis/AC-64.bvef'
montage = mne.channels.read_custom_montage(montage_path, head_size= 0.085)

### Epoch 

In [ ]:
epoch_list = {}

for participant in tqdm(df_filtered["Participant"].unique(), total= len(df_filtered["Participant"].unique())):
    print(f"\n Processing Participant {participant}")

    # load fif files
    eeg_fif_files = df_filtered[df_filtered["Participant"] == participant]["EEG"]

    participant_epochs=[]

    for eeg_file in eeg_fif_files:

        # Load and read eeg raw files
        raw = mne.io.read_raw_fif(eeg_file, preload= True)
        
        # Set montage
        raw.set_montage(montage)

        # Apply bandpass filter 
        raw.filter( 4, 50, fir_design = 'firwin')

        # Get raw channel data and perform common average referencing
        eeg_data_raw = raw.get_data()
        eeg_data_ref = eeg_data_raw - np.mean(eeg_data_raw, axis=0)

        # Extract channel_names of eeg signal
        channel_names = list(raw.to_data_frame().columns[1:])

        # Create temporal eeg raw for cutting data into epochs
        tmp_raw = mne.io.RawArray(eeg_data_ref, raw.info, verbose='ERROR')

        # Convert annotations to events
        events= np.array([(0,0,1)])

        # Considered minimumum duration of the Participant's code comprehension as the time window
        time_window = 4

        # Calculate total EEG signal duration
        eeg_duration = raw.n_times / raw.info['sfreq']

        # 
